# 新しいyamlのextractorを利用するパターン

In [1]:
# 必要ライブラリのインストール
# Note:: sslはデフォルトで入っているようでインストール不要(pipで入れようとするとエラーになる)
!pip install -U --quiet httpx
!pip install -U --quiet  openai
!pip install -U --quiet langchain-community langchain-openai langgraph langchain
!pip install -U --quiet python-dotenv

In [ ]:
import xml.etree.ElementTree as ET

from hourei_apiv2 import (
    extract_sections_from_xml,
    get_lawdata_from_law_id,
    get_lawid_from_lawtitle,
    save_xml_string_to_file,
)
from yaml_converter import convert_xml_to_yaml

# law_id = get_lawid_from_lawtitle("電気事業法")
# if law_id == None:
#     print("LAW DOES NOT EXISTS")


def extract_text_from_lawname(law_name: str, if_toc=True):
    law_id: str = get_lawid_from_lawtitle(law_name, if_exact=True)
    law_text: str = get_lawdata_from_law_id(law_id, "xml")
    # print(law_text)
    # save_xml_string_to_file(law_text, f"{law_title}.xml")

    # # xmlデータを読み込みます
    # with open(f"法令/{law_title}.xml") as f:
    #     law_text = f.read()
    # 一番上の階層の要素を取り出します
    # law_text = extract_sections_from_xml(law_text)
    law_text: str = convert_xml_to_yaml(law_text)
    return law_text

In [ ]:
law_name = "土壌汚染対策法"
law_text = extract_text_from_lawname(law_name)

regulation_name = "土壌汚染対策法施行規則"
regulation_text = extract_text_from_lawname(regulation_name)

ID: 414AC0000000053, Num: 平成十四年法律第五十三号, Title: 土壌汚染対策法
ID: 414CO0000000336, Num: 平成十四年政令第三百三十六号, Title: 土壌汚染対策法施行令
ID: 414M60001000023, Num: 平成十四年環境省令第二十三号, Title: 土壌汚染対策法に基づく指定調査機関及び指定支援法人に関する省令
ID: 414M60001000029, Num: 平成十四年環境省令第二十九号, Title: 土壌汚染対策法施行規則
Number of laws: 4
ID: 414M60001000029, Num: 平成十四年環境省令第二十九号, Title: 土壌汚染対策法施行規則
Number of laws: 1


In [ ]:
# OpenAI用の変数設定
import os
import ssl

import httpx
import openai
from dotenv import load_dotenv
from langchain_openai import AzureChatOpenAI, ChatOpenAI
from openai import AzureOpenAI, OpenAI

load_dotenv(verbose=True)
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")


def get_llm(modelname: str):

    # https://python.langchain.com/api_reference/openai/chat_models/langchain_openai.chat_models.base.ChatOpenAI.html
    reasoning = {
        "effort": "high",  # 'low', 'medium', or 'high'
        "summary": "auto",  # 'detailed', 'auto', or None
    }

    # Azureの呼び出しは基本同じ変数も言える
    # https://python.langchain.com/api_reference/openai/chat_models/langchain_openai.chat_models.azure.AzureChatOpenAI.html
    llm = ChatOpenAI(
        model=modelname,
        temperature=1.0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
        api_key=OPENAI_API_KEY,
        reasoning_effort="high",
        # base_url = "https://apim-tepcopgaiservice-dev-01.azure-api.net",
    )
    return llm


llm = get_llm("gpt-5")

# * 法令，施行規則の両方を読み混んで正解を出すパターン

もしもプロンプト長が事足りるならばこれが一番良い．
ここでも，抽出する際に与える文書をyaml形式で与えている


In [ ]:
# 法令の指定
from langchain.prompts import PromptTemplate, load_prompt

law_name = "土壌汚染対策法"
law_article = "第3条、４条"
enforcement_regulations_name = "土壌汚染対策法施行規則"
prompt_path = "prompts/v001.yaml"
# prompt_path = "prompts/extract_related_laws_v001.yaml"
law_text = extract_text_from_lawname(law_name)
enforcement_regulations = extract_text_from_lawname(enforcement_regulations_name)
prompt = load_prompt(prompt_path)
formatted_prompt = prompt.format(
    law_name=law_name,
    law_article=law_article,
    law_text=law_text,
    enforcement_regulations=enforcement_regulations,
)
messages = [
    ("system", formatted_prompt),
]
ai_msg = llm.invoke(messages)
ai_msg.pretty_print()

================================== Ai Message ==================================

### 官庁への申請が必要な場合
1. 面積3,000㎡以上の掘削・盛土・埋設など（法4条1項）
    - 工事の30日前までに都道府県へ届出が必要（規則23条）
    - しきい値：3,000㎡（規則22条）
    - 添付：平面図・立面図・断面図（規則23条2項）
    - 例）変電所造成、鉄塔多数の基礎造成で合計3,000㎡超

2. 「有害施設」の現・旧敷地で900㎡以上の形質変更（法4条1項）
    - 工事の30日前までに都道府県へ届出（規則23条）
    - しきい値：900㎡（規則22条ただし書）
    - 対象敷地：現に有害物質使用特定施設がある敷地／使用が廃止された同施設の敷地
      （ただし3条1項の報告済み地・確認済み地は除外：規則22条ただし書）
    - 添付：平面図・立面図・断面図（規則23条2項）
    - 例）旧メッキ工場跡地で鉄塔建替え、延べ900㎡以上の掘削

3. 3条の「確認」済みの土地で大きめの掘削等をする場合（法3条7項）
    - 工事着手前に都道府県へ届出（規則21条の2）
    - 届出不要の例外（規則21条の4）を満たさないときに必要
      - 面積900㎡未満は不要
      - 900㎡以上でも「土砂を場外搬出しない」「飛散・流出を伴わない」「深さ50cm未満」を全て満たせば不要
    - 添付：平面図・立面図・断面図（規則21条の2第2項）
    - 例）確認済みの土地で延べ1,000㎡・深さ1.5mの基礎撤去（土砂搬出あり）

### 注釈
1. 土地の形質の変更：地面の掘削・盛土・埋戻し・埋設などの行為
2. 届出先：都道府県知事（政令市は市長）（法・規則本文）
3. 届出期限：4条は着手30日前（法4条1項）。3条7項は「着手前」（規則21条の2）
4. しきい値：一般は3,000㎡、有害施設の現・旧敷地は900㎡（規則22条）
5. 3条「確認」済み土地の例外（届出不要条件）：面積900㎡未満、または900㎡以上でも「場外搬出なし・飛散流出なし・深さ50cm未満」を全て満たす場合（規則21条の4）


# 法令から最初に関連する条文を抽出する方法

v1との違いとして，以下の3手順を行う．
1. まずは法令全文を投入して，LLMから関連法令の一覧を取得
2. 関数で該当条項の全文を抽出
3. ついで施行規則全文を投入して，LLMから関連法令の一覧を取得
4. 関数で該当条項の全文を抽出
5. 2と4で抽出した全文を利用して判断軸を作成


2,3をLawExtracterで実行．

In [4]:
import logging
from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from enum import Enum
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional, TypedDict

import yaml
from langchain_core.language_models import BaseLLM
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph

from law_extraction import LegalDocument
from law_extraction_v2 import LegalExtractionConfig, create_legal_extraction_system

# ロギング設定
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

config = LegalExtractionConfig(llm=llm, prompts_dir="prompts")

# システムの初期化
extraction_system = create_legal_extraction_system(config)

# 法律名指定
# ======================================
law_name = "土壌汚染対策法"
target_articles = ["第3条", "第4条"]
# ======================================

# 法令
law_text = extract_text_from_lawname(law_name)
law_yaml = yaml.safe_load(law_text)
law_document = LegalDocument(
    name=law_name, content=law_text, document_type="law", yaml_data=law_yaml
)

# 施行規則
regulation_name = law_name + "施行規則"
regulation_text = extract_text_from_lawname(regulation_name)
regulation_yaml = yaml.safe_load(regulation_text)
regulation_document = LegalDocument(
    name=regulation_name,
    content=regulation_text,
    document_type="regulation",
    yaml_data=regulation_yaml,
)

# 処理の実行
try:
    result = extraction_system.process(
        law_document=law_document,
        regulation_document=regulation_document,
        target_articles=target_articles,
    )

    if result["error_message"]:
        print(f"エラーが発生しました: {result['error_message']}")
    else:
        print("=== 最終要点 ===")
        print(result["final_summary"])

        print("\n=== メタデータ ===")
        for key, value in result["metadata"].items():
            print(f"{key}: {value}")

except Exception as e:
    logger.error(f"処理中に予期しないエラーが発生しました: {e}")

INFO:law_extraction_v2:法令判断軸抽出処理を開始します
INFO:law_extraction_v2:法令からの関連条文抽出を開始
INFO:law_extraction_v2:LLMによる関連条文番号の特定を開始


debug subset = {'law_article': '第3条, 第4条', 'law_name': '土壌汚染対策法', 'law_text': "law_info:\n  title: 土壌汚染対策法\ntable_of_contents:\n- type: label\n  content: 目次\n- type: chapter\n  title: 第一章\u3000総則\n  article_range: （第一条・第二条）\n- type: chapter\n  title: 第二章\u3000土壌汚染状況調査\n  article_range: （第三条―第五条）\n- type: chapter\n  title: 第五章\u3000指定調査機関\n  article_range: （第二十九条―第四十三条）\n- type: chapter\n  title: 第六章\u3000指定支援法人\n  article_range: （第四十四条―第五十三条）\n- type: chapter\n  title: 第七章\u3000雑則\n  article_range: （第五十四条―第六十四条）\n- type: chapter\n  title: 第八章\u3000罰則\n  article_range: （第六十五条―第六十九条）\n- type: supplementary\n  content: 附則\nchapters:\n- title: 第一章\u3000総則\n  chapter_num: 1\n  articles:\n  - caption: （目的）\n    title: 第一条\n    article_num: '1'\n    paragraphs:\n    - content: この法律は、土壌の特定有害物質による汚染の状況の把握に関する措置及びその汚染による人の健康に係る被害の防止に関する措置を定めること等により、土壌汚染対策の実施を図り、もって国民の健康を保護することを目的とする。\n  - caption: （定義）\n    title: 第二条\n    article_num: '2'\n    paragraphs:\n    - content: この法律において「特定有害物質」とは、鉛、砒（

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:law_extraction_v2:特定された関連条文: ['3_1', '16_1', '6_1', '17', '18_1', '22_1', '22_2', '22_3', '22_4', '25', '23_1']
INFO:law_extraction_v2:YAML構造から11件の条文を抽出中
ERROR:law_extraction_v2:条文3_1の抽出でエラー: 第3_1条が見つかりません
ERROR:law_extraction_v2:条文16_1の抽出でエラー: 第16_1条が見つかりません
ERROR:law_extraction_v2:条文6_1の抽出でエラー: 第6_1条が見つかりません
ERROR:law_extraction_v2:条文18_1の抽出でエラー: 第18_1条が見つかりません
ERROR:law_extraction_v2:条文22_1の抽出でエラー: 第22_1条が見つかりません
ERROR:law_extraction_v2:条文22_2の抽出でエラー: 第22_2条が見つかりません
ERROR:law_extraction_v2:条文22_3の抽出でエラー: 第22_3条が見つかりません
ERROR:law_extraction_v2:条文22_4の抽出でエラー: 第22_4条が見つかりません
ERROR:law_extraction_v2:条文23_1の抽出でエラー: 第23_1条が見つかりません
INFO:law_extraction_v2:法令抽出が完了しました
INFO:law_extraction_v2:施行規則からの関連条文抽出を開始
INFO:law_extraction_v2:LLMによる施行規則の関連条文番号の特定を開始


debug subset = {'extracted_law_content': ' 法令本文：抽出された関連条項\n\n【注意】第3_1条の内容を取得できませんでした\n\n\n【注意】第16_1条の内容を取得できませんでした\n\n\n【注意】第6_1条の内容を取得できませんでした\n\n\n第17条（（運搬に関する基準）） 第十七条\n要措置区域等外において汚染土壌を運搬する者は、環境省令で定める汚染土壌の運搬に関する基準に従い、当該汚染土壌を運搬しなければならない。 ただし、非常災害のために必要な応急措置として当該運搬を行う場合は、この限りでない。\n\n\n【注意】第18_1条の内容を取得できませんでした\n\n\n【注意】第22_1条の内容を取得できませんでした\n\n\n【注意】第22_2条の内容を取得できませんでした\n\n\n【注意】第22_3条の内容を取得できませんでした\n\n\n【注意】第22_4条の内容を取得できませんでした\n\n\n第25条（（許可の取消し等）） 第二十五条\n都道府県知事は、汚染土壌処理業者が次の各号のいずれかに該当するときは、その許可を取り消し、又は一年以内の期間を定めてその事業の全部若しくは一部の停止を命ずることができる。\n  一 第二十二条第三項第二号イ又はハからトまでのいずれかに該当するに至ったとき。\n  二 汚染土壌処理施設又はその者の能力が第二十二条第三項第一号の環境省令で定める基準に適合しなくなったとき。\n  三 この章の規定又は当該規定に基づく命令に違反したとき。\n  四 不正の手段により第二十二条第一項の許可（同条第四項の許可の更新を含む。）又は第二十三条第一項の変更の許可を受けたとき。\n\n\n【注意】第23_1条の内容を取得できませんでした\n', 'law_name': '土壌汚染対策法', 'regulation_text': "law_info:\n  title: 土壌汚染対策法施行規則\narticles:\n- caption: （使用が廃止された有害物質使用特定施設に係る工場又は事業場の敷地であった土地の調査）\n  title: 第一条\n  article_num: '1'\n  paragraphs:\n  - content: 土壌汚染対策法（平成十四年法律第五十三

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:law_extraction_v2:特定された関連条文: ['1', '2', '3', '3_2', '4', '5', '6', '7', '8', '9', '10', '10_2', '10_3', '11', '12', '13', '13_2', '14', '14_2', '15', '16', '17', '18', '19', '20', '21', '21_2', '21_3', '21_4', '21_5', '21_6', '31', '32', '58', '59', '59_2', '59_3', '60', '61', '62', '63', '64', '65', '65_2', '65_3', '65_4', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '76_2', '77']
INFO:law_extraction_v2:施行規則のYAML構造から59件の条文を抽出中
INFO:law_extraction_v2:施行規則抽出が完了しました
INFO:law_extraction:最終的な判断軸生成を開始


debug subset = {'enforcement_regulations': ' 施行規則：抽出された関連条項\n\n第1条（（使用が廃止された有害物質使用特定施設に係る工場又は事業場の敷地であった土地の調査）） 第一条\n土壌汚染対策法（平成十四年法律第五十三号。以下「法」という。）第三条第一項本文の報告は、次の各号に掲げる場合の区分に応じ、当該各号に定める日から起算して百二十日以内に行わなければならない。 ただし、当該期間内に当該報告を行うことができない特別の事情があると認められるときは、都道府県知事（土壌汚染対策法施行令（平成十四年政令第三百三十六号。以下「令」という。）第十条に規定する市にあっては、市長。以下同じ。）は、当該土地の所有者等（法第三条第一項本文に規定する所有者等をいう。以下同じ。）の申請により、その期限を延長することができる。\n  一 当該土地の所有者等が当該有害物質使用特定施設（法第三条第一項に規定する有害物質使用特定施設をいう。以下同じ。）を設置していた者である場合（同項ただし書の確認を受けた場合を除く。） 当該有害物質使用特定施設の使用が廃止された日\n  二 当該土地の所有者等が法第三条第三項の通知を受けた者である場合（法第三条第一項ただし書の確認を受けた場合を除く。） 当該通知を受けた日\n  三 法第三条第一項ただし書の確認が取り消された場合 第二十一条の通知を受けた日\n（第２項）\n法第三条第一項本文の報告は、次に掲げる事項を記載した様式第一による報告書を提出して行うものとする。\n  一 氏名又は名称及び住所並びに法人にあっては、その代表者の氏名\n  二 工場又は事業場の名称及び当該工場又は事業場の敷地であった土地の所在地\n  三 使用が廃止された有害物質使用特定施設の種類、設置場所及び廃止年月日並びに当該有害物質使用特定施設において製造され、使用され、又は処理されていた特定有害物質（法第二条第一項に規定する特定有害物質をいう。以下同じ。）の種類その他の土壌汚染状況調査（同条第二項に規定する土壌汚染状況調査をいう。以下同じ。）の対象となる土地（以下「土壌汚染状況調査の対象地」という。）において土壌の汚染状態が第三十一条第一項の基準（以下「土壌溶出量基準」という。）又は同条第二項の基準（以下「土壌含有量基準」という

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:law_extraction_v2:要点生成が完了しました
INFO:law_extraction_v2:処理が完了しました。ステージ: ProcessingStage.COMPLETED


=== 最終要点 ===
viewpoint='### 官庁への申請が必要な場合\n1. 3条1項ただし書確認土地で50cm以上掘削\n    - 面積900㎡以上の掘削は要届出（法3条7項、施行規則21条の四）\n2. 3条1項ただし書確認土地で土砂を区域外搬出\n    - 面積900㎡以上で搬出は要届出（法3条7項、施行規則21条の四イ）\n3. 3条1項ただし書確認土地で粉じん・流出のおそれ\n    - 面積900㎡以上で発じん等は要届出（法3条7項、施行規則21条の四ロ）' annotation='### 注釈\n1. 土地の形質の変更＝地面を掘る・盛る・造成する等の工事。\n2. 「3条1項ただし書確認土地」＝人の健康被害が生じない旨の確認（法3条1項ただし書）を受けた土地（申請様式は施行規則16条）。\n3. 面積900㎡未満の工事は届出不要（施行規則21条の四1号）。\n4. 面積900㎡以上でも、深さ50cm未満・区域外搬出なし・粉じん/流出なしなら届出不要（施行規則21条の四2号）。\n5. 上記の届出は法3条7項に基づくもので、電柱・鉄塔基礎の大規模掘削や大量搬出が該当しやすい。'

=== メタデータ ===
stage: summary_generation
source_document: 土壌汚染対策法施行規則
target_articles: ['第3条', '第4条']
identified_articles: ['1', '2', '3', '3_2', '4', '5', '6', '7', '8', '9', '10', '10_2', '10_3', '11', '12', '13', '13_2', '14', '14_2', '15', '16', '17', '18', '19', '20', '21', '21_2', '21_3', '21_4', '21_5', '21_6', '31', '32', '58', '59', '59_2', '59_3', '60', '61', '62', '63', '64', '65', '65_2', '65_3', '65_4', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '76

In [ ]:
print(" -------- view point ------------- ")
print(result["final_summary"].viewpoint)

print(" -------- annotation ------------- ")
print(result["final_summary"].annotation)

 -------- view point ------------- 
### 官庁への申請が必要な場合
1. 一般地で3,000㎡以上の掘削・盛土等（法4条1項、規則22・23・24・25）
    - 着手30日前までに届出
2. 特定施設の敷地等で900㎡以上の掘削・盛土等（法4条1項、規則22・23・24・25）
    - 着手30日前までに届出
3. 3条確認土地での形質変更（規則21条の4の例外除く）（法3条7項、規則21条の2・21条の3・21条の4）
    - 工事前に場所等を届出
 -------- annotation ------------- 
### 注釈
1. 土地の形質の変更とは
   - 掘削・盛土・埋戻し・杭打ち等（電柱・鉄塔の基礎工事を含む）
2. 4条の規模基準（規則22条）
   - 一般の土地：3,000㎡以上で届出
   - 特定施設の敷地等：900㎡以上で届出
   - 3条確認土地は4条の対象外（法4条1項ただし書一）
3. 4条の届出期限・様式（法4条1項、規則23・24）
   - 期限：着手30日前まで
   - 届出書：様式第六（規則23条）
   - 添付：平面図・立面図・断面図（規則23条2項一）
   - 記載事項：所在地・面積・深さ等（規則24条）
4. 4条の届出不要（軽易な行為）（規則25条）
   - 土砂を区域外へ搬出しない
   - 土砂の飛散・流出を伴わない
   - 掘削深さが50cm未満
   - 農業の通常行為、林業の作業路網整備
   - 鉱山関係の土地
   - 知事が安全と指定した土地
5. 3条確認土地の事前届出（法3条7項、規則21条の2・21条の3）
   - 届出書：様式第六（規則21条の2）
   - 添付：平面図・立面図・断面図（規則21条の2第2項）
   - 記載事項：所在地・面積・深さ等（規則21条の3）
6. 3条確認土地で届出不要（規則21条の4、法3条7項二）
   - 面積が900㎡未満
   - 面積900㎡以上でも次の全て満たせば不要：
     ・土砂の区域外搬出なし／飛散・流出なし／深さ50cm未満
   - 非常災害の応急措置は不要
7. 用語の意味
   - 有害物質使用特定施設：水濁法2条2項の特定施設で特定有害物質を製造・使用・処理す